# 🛣️ RoadSense — training notebook

Runs the full pipeline on a free Colab GPU:
1. download RDD2022 (India) and convert to YOLO format
2. train a **baseline** model, then an **improved** model (both logged to MLflow)
3. evaluate on the held-out test split + error analysis
4. download `best.pt` and `metrics.json` for the backend

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

In [ ]:
!nvidia-smi -L
!git clone https://github.com/tejasavbishnoi/roadsense.git
%cd roadsense/training
!pip install -q -r requirements.txt

## 1. Prepare the dataset
Downloads ~500 MB. Add `--extra /content/pune` after uploading your own Roboflow export (YOLOv8 format) to include your Pune photos.

In [ ]:
!python prepare_data.py --countries India --out ../datasets/roadsense

## 2. Baseline (≈20–30 min on a T4)

In [ ]:
!python train.py --preset baseline --device 0 --export-to ''

## 3. Improved model (≈1–1.5 h)
Bigger model + road-specific augmentation. Copies `best.pt` to `backend/weights/`.

In [ ]:
!python train.py --preset improved --device 0

## 4. Evaluate on the test split

In [ ]:
!python evaluate.py --weights ../backend/weights/best.pt

Compare the two runs:

In [ ]:
import json, glob
for f in sorted(glob.glob('runs/*/run_summary.json')):
    s = json.load(open(f))
    m = s['metrics']
    print(f"{s['run']:10s} {s['model']:12s} mAP50={m.get('metrics/mAP50(B)',0):.3f}  mAP50-95={m.get('metrics/mAP50-95(B)',0):.3f}")

Look at where it fails:

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('results/test/worst_images/*.jpg'))[:6]:
    display(Image(p, width=640))

## 5. Download the results
Put `best.pt` and `metrics.json` into `backend/weights/` on your laptop.

In [ ]:
!zip -qr /content/roadsense_results.zip ../backend/weights results runs/*/run_summary.json runs/*/results.png runs/*/confusion_matrix*.png mlruns
from google.colab import files
files.download('/content/roadsense_results.zip')